# Final pipeline

## Getting raw files (test set)

In [1]:
!pip install yadisk -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.7/140.7 kB 8.0 MB/s eta 0:00:00


In [2]:
import yadisk
import os
import zipfile

In [3]:
# инициализация клиента и ссылка
yd = yadisk.YaDisk()
public_link = "https://disk.yandex.ru/d/kIlHLgOMYgA7gA"

try:
    # ~ 1-5 минут. размер: ~3,9 гб
    yd.download_public(
        public_link,
        'test_set.zip'
        )
    print('completed')
except Exception as e:
    print(e)

completed


In [4]:
!unzip -q /content/test_set.zip -d /content/test_set

## Dependencies

In [5]:
!pip -q install "stable-ts[fw]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.1/189.1 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 3.0 MB/s eta 0:00:00


In [6]:
import torch
import os
import json

import librosa
from stable_whisper import load_faster_whisper

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import re
from datetime import date
from tqdm.notebook import tqdm

from transformers import AutoTokenizer, AutoModel

## Funcs to run

In [7]:
# сгруппировать файлы в подпакеты "оператор-год"
def group_agents_years(file_names: list) -> dict:
    agents_years_dict = {}

    for file_name in file_names:
        timestamp, agent_id = file_name.split('_', 1)

        # номер оператора в формате `_01` (str). так согласовано
        agent_id = agent_id.rsplit('.', 1)[0].\
                  replace('.', '').\
                  removesuffix('_muted').\
                  removeprefix('Оператор')
        year = timestamp[:4]
        agent_id = f'{agent_id.zfill(2)}_{year}'

        # добавляем в словарь
        agents_years_dict[agent_id] = agents_years_dict.setdefault(agent_id, [])
        agents_years_dict[agent_id].append(file_name)

    return agents_years_dict

In [8]:
def get_raw_timeline(model,
                     file_path
                     ) -> list:

    # каналы из файла: 1 = оператор
    # только 16000, не моно; иначе виспер не справится
    y, sr = librosa.load(file_path, sr=16000, mono=False)
    agent_audio = y[1]

    # промпт для адаптации под домен техподдержки с паравербаликой
    tech_support_prompt = (
        "Разговор клиента с оператором техподдержки "
        "(IP-адрес, роутер, интернет, сеть, логин, компьютер, пароли). "
        "Участники вздыхают, запинаются и используют "
        "междометия в речи, например: эээ, ммм, ага, угу, так, ой. "
        "В тексте сохраняются все паузы, вздохи, цифры и термины."
    )
    # параметры сырой транскрибации. подобраны эмпирически
    transcribe_params = dict(
        language='ru',
        initial_prompt=tech_support_prompt,
        suppress_tokens=[-1], # не блокируем токены, чтобы не терять вздохи и др.
        word_timestamps=True, # временные метки слов для расчета метрик
        vad_filter=True,
        vad_parameters=dict(
            min_speech_duration_ms=250, # не менее 250 миллисекунд на реплику
            max_speech_duration_s=15, # есть весьма длинные
            speech_pad_ms=600
            ),
        temperature=[0.0, 0.2, 0.4, 0.6],
        compression_ratio_threshold=2.4,
        no_speech_threshold=0.8 # чтобы отлавливать тихие звуки (уверенность в том, что сегмент не содержит речи, = 0,8, т.е. весьма высокая). артефакты потом чистим регулярками
    )

    # транскрибация из канала оператора
    agent_result = model.transcribe(agent_audio, **transcribe_params)

    # первичный парсинг
    raw_timeline = []
    for seg in agent_result.segments:
        # временные метки при наличии
        words_data = []
        if hasattr(seg, 'words') and seg.words:
            words_data = [{'word': w.word, 'start': w.start, 'end': w.end} for w in seg.words]

        raw_timeline.append({
            'start': seg.start,
            'end': seg.end,
            'text': seg.text.strip(),
            'words': words_data
        })

    # сортировка чанков по временным меткам (с одним спикером уже излишне, скорее всего)
    raw_timeline.sort(key=lambda x: x['start'])

    return raw_timeline, agent_audio

In [9]:
# объединение последовательных реплик спикера, если между ними не более 1,2 секунды
# обычно в русском языке мена ролей в диалоге происходит при паузе от 1 до 1,5 секунды
# в наборе из-за пустот между репликами возможны паузы до нескольких минут
def merge_timeline(raw_timeline,
                   MAX_MERGE_GAP_SEC=1.2 # порог объединения
                   ) -> list:

    merged_timeline = []
    if raw_timeline:
        current = raw_timeline[0]

        for next_seg in raw_timeline[1:]:
            # пауза между сегментами
            gap_between_segments = next_seg['start'] - current['end']

            # сливаем реплики, если пауза не больше порога
            if gap_between_segments <= MAX_MERGE_GAP_SEC:
                current['end'] = next_seg['end']
                current['text'] += " " + next_seg['text']
                current['words'].extend(next_seg['words'])

            # иначе переходим к следующему сегменту
            else:
                merged_timeline.append(current)
                current = next_seg

        merged_timeline.append(current)

    return merged_timeline

In [10]:
# подсчет метрик по реплике
def process_merged_timeline(merged_timeline,
                            agent_audio,
                            text_embedder,
                            ### OPENSMILE_PARSER, ###
                            SPLITTER_MS=400,
                            ) -> pd.DataFrame:
    # фильтрация галлюцинаций транскрибатора. могут быть и другие
    hallucination_pattern = re.compile(
        r'(прис|продолжение\s+следует|субтитры\s+сделал|спасибо\s+за\s+субтитры\s+алексею\s+дубровскому|субтитры\s+создавал|dimatorzok|подписывайтесь\s+на\s+канал|спасибо\s+за\s+просмотр)',
        re.IGNORECASE
        )

    # для сводных аудиодорожек и транскрипций
    rep_arrays_list = []
    rep_texts_list = []

    # для расчета паузации и скорости речи по репликам
    rep_interword_pause_means = []
    rep_speech_rates = []

    rows = []
    for i, rep in enumerate(merged_timeline):
        # очистка от галлюцинаций
        cleaned_text = hallucination_pattern.sub("", rep['text']).strip()
        cleaned_text = re.sub(r'^[\s\.\,\-\?\!]+$', "", cleaned_text).strip()

        if not cleaned_text:
            continue

        # длительность в миллисекундах
        rep_duration_ms = int((rep['end'] - rep['start']) * 1000)
        rep_duration_sec = rep_duration_ms / 1000.0
        rep_word_count = len(cleaned_text.split()) if cleaned_text else 0

        # скорость речи в реплике = слов в секунду. обычно в русском языке 2-3,5 слов в секунду
        rep_speech_rate_wps = round(rep_word_count / rep_duration_sec, 2) if rep_duration_sec > 0 else 0.0
        rep_speech_rates.append(rep_speech_rate_wps)

        # средняя длительность паузы между словами в реплике
        rep_interword_pauses = []
        words = rep['words']
        if words and len(words) > 1:
            for idx in range(len(words) - 1):
                pause = words[idx+1]['start'] - words[idx]['end']
                if pause > 0:
                    rep_interword_pauses.append(pause)
        if rep_interword_pauses:
            rep_interword_pause_means.append(np.mean(rep_interword_pauses))

        # вырезание реплики
        start_idx = int(rep['start'] * 16000)
        end_idx = int(rep['end'] * 16000)
        rep_audio_chunk = agent_audio[start_idx:end_idx]

        # слияние всех реплик
        rep_arrays_list.append(rep_audio_chunk)
        rep_texts_list.append(cleaned_text)

    # сплиттер между фрагментами. аналог в транскрипции представлен как `|`.
    splitter = np.zeros(SPLITTER_MS * int(16000 / 1000), dtype=int)
    agent_result_list = []
    for i, arr in enumerate(rep_arrays_list):
        agent_result_list.append(arr)
        # аналогично join
        if i < len(rep_arrays_list) - 1:
            agent_result_list.append(splitter)

    # сводные аудиодорожка и транскрипция
    # ! канал может быть пустым (не распознано реплик), тогда конкатенация
    # выдаст ValueError. в пайплайне обработано через try/except.
    agent_result_audio = np.concatenate(agent_result_list)
    agent_result_text = ' | '.join(rep_texts_list)

    # пауза между словами. из средних по репликам: среднее + станд. откл.
    avg_interword_pause_ms = int(np.mean(rep_interword_pause_means) * 1000) if rep_interword_pause_means else 0
    std_interword_pause_ms = int(np.std(rep_interword_pause_means) * 1000) if rep_interword_pause_means else 0

    # RMS энергии
    rms_energy = np.sqrt(np.mean(agent_result_audio**2)) if len(agent_result_audio) > 0 else 0

    # от средней скорости речи для каждой реплики: среднее + станд. откл.
    avg_speech_rate_wps = round(np.mean(rep_speech_rates), 2) if rep_speech_rates else 0
    std_speech_rate_wps = round(np.std(rep_speech_rates), 2) if rep_speech_rates else 0

    # количество слов в сводной транскрипции (включая предлоги итд)
    word_count = len(agent_result_text.replace('|', '').split()) if agent_result_text else 0

    # длительность сводной аудиодорожки
    duration_sec = len(agent_result_audio) / 16000

    # средняя скорость речи по сводной аудиодорожке
    acc_speech_rate_wps = round(word_count / duration_sec, 2) if duration_sec > 0 else 0.0

    # первичные признаки
    agent_result_dict = {
        # 'audio': agent_result_audio,
        'text': agent_result_text,
        'word_count': int(word_count),
        'avg_interword_pause_ms': float(avg_interword_pause_ms),
        'std_interword_pause_ms': float(std_interword_pause_ms),
        'avg_speech_rate_wps': float(avg_speech_rate_wps),
        'std_speech_rate_wps': float(std_speech_rate_wps),
        'acc_speech_rate_wps': float(acc_speech_rate_wps),
        'duration_sec': float(duration_sec),
        'rms_energy': float(round(float(rms_energy), 6))
        }

    # эмбеддинги из BERT (по умолчанию -- RoSBERTa, 1024)
    bert_embeddings = text_embedder.extract(agent_result_text)
    bert_feature_names = [f'bert_{i + 1}' for i in range(len(bert_embeddings))]
    bert_dict = dict(zip(bert_feature_names, bert_embeddings))
    agent_result_dict |= bert_dict

    # !! OpenSMILE
    ### OPENSMILE_PARSER.parse(agent_result_audio) ###

    return agent_result_dict

In [11]:
# парсим название файла
def parse_file_path(file_path) -> tuple:
    file_name = file_path.rsplit('/', 1)[-1]
    file_timestamp, agent_id = file_name.split('_', 1)

    # номер оператора в формате `_01` (str). так согласовано
    agent_id = agent_id.rsplit('.', 1)[0].\
               replace('.', '').\
               removesuffix('_muted').\
               removeprefix('Оператор')
    agent_id = f'_{agent_id.zfill(2)}'

    # год, месяц, дата, час записи (int)
    year = int(file_timestamp[:4])
    month = int(file_timestamp[4:6])
    day = int(file_timestamp[6:8])
    hour = int(file_timestamp[8:10])

    # день недели (str)
    weekday = date(int(year), int(month), int(day)).strftime("%a")

    # анные для новых столбцов
    new_cols = (file_name, agent_id, year, month, day, weekday, hour)

    return new_cols

In [12]:
# дополнение словаря
def process_dict(agent_result_dict, file_path):

    # столбцы из старого имени для отладки. ставятся в начало
    new_cols = parse_file_path(file_path)
    ID_dict = dict(zip(
        ['file_name', 'agent_id', 'year', 'month', 'day', 'weekday', 'hour'],
        new_cols)
    )
    new_dict = ID_dict | agent_result_dict

    return new_dict

In [13]:
def predict(row: pd.Series, # строка датафрейма
            JSON_STRING
            ):
    # выгружаем json в словарь
    _secret_dict = JSON_STRING#json.loads(JSON_STRING)

    # предсказываем: свободный член + сумма произведений
    # извлекаем свободный член
    prediction_score = _secret_dict['Intercept']

    # определяем пересечение с присутствующими признаками:
    # ключи словаря & индекс строки
    feature_set = set(_secret_dict.keys()).intersection(row.index)
    # print(len(feature_set))

    for feat in feature_set:
        # забираем вес признака
        weight = _secret_dict[feat]

        # добавляем произведение к свободному члену
        prediction_score += row[feat] * weight

    return prediction_score

In [14]:
def predict_all(agent_year_df,
                json_string_filter,
                json_strings_experts,
                json_string_agent_sex,
                json_string_burnout,
                ):

    filtered_rows = []
    for i, row in agent_year_df.iterrows():
        # предсказание №1 = фильтрация
        # оценить зашумленность/малоинформативность (чем больше, тем хуже)
        filter_score = predict(row, json_string_filter)
        # print(i, filter_score)
        threshold = 0.5

        # оставляем только те, что ниже порога
        if round(filter_score, 8) < threshold:
            # предсказание №2 (серия из 8 признаков) = психолингвистика
            # предсказываем все 8 реконструированных признаков экспертной разметки
            for expert_feat, json_string_expert in json_strings_experts.items():
                row[expert_feat] = predict(row, json_string_expert)

            # добавляем в отфильтрованный набор обогащенную строку
            filtered_rows.append(row)

    # объединение отфильтрованных и обогащенных
    agent_year_df = pd.DataFrame(filtered_rows)

    # предсказание №3. пол
    json_string_agent_sex

    # индекс столбца с текстом + 1, чтобы взять только числовые столбцы после него
    agent_cols = agent_year_df.columns.tolist()
    agent_text_ix = agent_cols.index('text')

    # усреднение по пакету "оператор-год" и предсказание пола
    agent_year_mean = agent_year_df[agent_cols[agent_text_ix + 1:]].mean()
    agent_sex_predicted = predict(agent_year_mean, json_string_agent_sex)
    # бинаризация для дальнейших предсказаний
    agent_sex_predicted = int(round(agent_sex_predicted, 8) > 0.5)

    agent_year_mean['m'] = agent_sex_predicted
    agent_year_mean['f'] = 1 - agent_sex_predicted

    agent_year_df['m'] = agent_sex_predicted
    agent_year_df['f'] = 1 - agent_sex_predicted

    # agent_year_mean['agent_sex'] = agent_sex_predicted
    # agent_year_df['agent_sex'] = agent_sex_predicted

    # предсказание №4. "выгорание"
    json_string_burnout
    # по средним признаков
    agent_burnout_predicted = predict(agent_year_mean, json_string_burnout)
    agent_year_mean['burnout'] = agent_burnout_predicted
    agent_year_df['burnout'] = agent_burnout_predicted
    # бинаризация по порогу 0.5
    agent_year_df['burnout_final'] = (agent_year_df['burnout'].round(8) > 0.5).astype(int)

    # сокращенная версия с предсказаниями; полная -- для интерпретации
    return (agent_year_df[['file_name', 'burnout', 'burnout_final']],
            agent_year_df)

## BERT for text embeddings

In [15]:
class TextEmbedder:
    # по умолчанию -- "ai-forever/ru-en-RoSBERTa" для максимального качества семантики
    # хотя мб хватило бы "cointegrated/rubert-tiny2"
    def __init__(self,
                 model_name="ai-forever/ru-en-RoSBERTa",
                 device="cuda",
                 ):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device).eval()
        self.embedding_size = self.model.config.hidden_size # 312 для rubert/tiny2, 1024 для RoSBERTa
        self.device = device

    @torch.no_grad()
    def extract(self, text):
        if not isinstance(text, str) or len(text.strip()) == 0:
            return np.zeros(self.embedding_size, dtype=np.float32)

        # Для тяжелых моделей важно установить max_length и truncation
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        ).to(self.device)

        outputs = self.model(**inputs)

        # Mean pooling по токенам с учетом attention_mask
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = inputs.attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

        return (sum_embeddings / sum_mask).squeeze(0).cpu().numpy()

## Loading models

In [16]:
# проверка GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [17]:
model = load_faster_whisper(
    'large-v3-turbo',
    # 'large-v3',
    device=device,
    compute_type='float16'
    )

In [18]:
text_embedder = TextEmbedder(
    model_name="ai-forever/ru-en-RoSBERTa",
    device=device
    )

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.49M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/5.99M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.61GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: ai-forever/ru-en-RoSBERTa
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Inference loop

In [19]:
# подтянуть JSON с коэффициентами

### JSON с коэффициентами из переменных окружения ###
with open('json_string_filter.json', 'r') as f:
    json_string_filter = json.load(f)

with open('json_string_participation_modal_n.json', 'r') as f:
    json_string_participation_modal_n = json.load(f)

with open('json_string_greeting_flg.json', 'r') as f:
    json_string_greeting_flg = json.load(f)

with open('json_string_farewell_flg.json', 'r') as f:
    json_string_farewell_flg = json.load(f)

with open('json_string_etiquette_present_flg.json', 'r') as f:
    json_string_etiquette_present_flg = json.load(f)

with open('json_string_etiquette_absent_flg.json', 'r') as f:
    json_string_etiquette_absent_flg = json.load(f)

with open('json_string_participation_dominant_flg.json', 'r') as f:
    json_string_participation_dominant_flg = json.load(f)

with open('json_string_distancing_dominant_flg.json', 'r') as f:
    json_string_distancing_dominant_flg = json.load(f)

with open('json_string_clarifying_absent_flg.json', 'r') as f:
    json_string_clarifying_absent_flg = json.load(f)

with open('json_string_agent_sex.json', 'r') as f:
    json_string_agent_sex = json.load(f)

with open('json_string_burnout.json', 'r') as f:
    json_string_burnout = json.load(f)


# для экспертной разметки (8 признаков)
json_strings_experts = {
    'participation_modal_n': json_string_participation_modal_n,

    'greeting_flg': json_string_greeting_flg,
    'farewell_flg': json_string_farewell_flg,

    'etiquette_present_flg': json_string_etiquette_present_flg,
    'etiquette_absent_flg': json_string_etiquette_absent_flg,

    'participation_dominant_flg': json_string_participation_dominant_flg,

    'distancing_dominant_flg': json_string_distancing_dominant_flg,
    'clarifying_absent_flg': json_string_clarifying_absent_flg,
}

In [20]:
# откуда берем файлы
input_dir = '/content/test_set'
input_files = os.listdir(input_dir)

file_dicts = []
input_dir

'/content/test_set'

In [21]:
# сгруппировать в подпакеты "оператор-год"
agents_years_dict = group_agents_years(input_files)

# словарь для хранения датафреймов с признаками
agent_years_df_dict = {}

# итерируемся по словарю: сводный ID оператор-год и [все его файлы за этот год]
for agent_year_id, files_to_process in agents_years_dict.items():

    # список для хранения словарей по всем записям для оператор/год
    agent_dicts = []

    # итерируемся по всем файлам этого "оператор-год"
    for i, original_file in enumerate(files_to_process, start=1):
        # путь к файлу
        file_path = os.path.join(
            input_dir,
            original_file
            )
        # для быстрой отладки
        # print(f'Оператор_Год={agent_year_id}: {i}/{len(files_to_process)}. {original_file}'.center(80, '='))

        # пайплайн
        try:
            # 1. получаем временные метки чанков и массив аудио из канала агента
            raw_timeline, agent_audio = get_raw_timeline(
                model,
                file_path
                )
            # 2. объединяем чанки в цельные реплики (не более 1.2 секунд между ними)
            merged_timeline = merge_timeline(
                raw_timeline,
                1.2 # предельная длительность паузы, при которой чанки сливаются
                )
            # 3. получаем дорожку без пустот и словарь с транкрипцией и метриками
            agent_result_dict = process_merged_timeline(
                merged_timeline,
                agent_audio,
                text_embedder,

                ### OPENSMILE_PARSER, ###

                SPLITTER_MS=400 # длительность сплиттера в миллисекундах
                )
            # 4. дополняем словарь и формируем имя для нового аудиофайла
            new_dict = process_dict(
                agent_result_dict,
                file_path,
                )

            # 5. результаты в виде словаря -> список словарей
            agent_dicts.append(new_dict)

        # если не распозналось ни одной реплики, то упадет с ошибкой.
        # пропускаем такой файл, их не слишком много
        except Exception as e:
            # принт для отладки
            # print(f'Failed: {e} ({file_path})')
            continue

    # сложить словари в сводный датафрейм и отсортировать по хронологии
    agent_year_df = pd.DataFrame(agent_dicts)
    agent_year_df = agent_year_df.sort_values(
        ['agent_id', 'year', 'month', 'day', 'hour']
        ).reset_index(drop=True)

    # добавить датафрейм в словарь по сводному ID
    agent_years_df_dict[agent_year_id] = agent_year_df

# список для предсказаний по отдельным "оператор-год"
agent_years_df_predicted_list = []
agent_years_df_full_list = []

# 6. все предсказания
for agent_year_id, agent_year_df in agent_years_df_dict.items():
    agent_year_predicted_df, agent_year_df_full = predict_all(
        agent_year_df,
        json_string_filter,
        json_strings_experts,
        json_string_agent_sex,
        json_string_burnout
        )
    agent_years_df_predicted_list.append(agent_year_predicted_df)
    agent_years_df_full_list.append(agent_year_df_full)

# итоговый датафрейм
final_predictions_df = pd.concat(
    agent_years_df_predicted_list, axis=0
    )

# полный датафрейм для интерпретации
final_interpretation_df = pd.concat(
    agent_years_df_full_list, axis=0
    )

# сохранение в таблицу (xlsx)
final_predictions_df.to_excel('result.xlsx', index=False)
final_interpretation_df.to_excel('result_full.xlsx', index=False)

Detected Language: russian


Transcribe: 100%|██████████| 390.61/390.61 [00:05<00:00, 67.15sec/s]
Adjustment: 100%|██████████| 387.84/387.84 [00:00<00:00, 51489.85sec/s]


Detected Language: russian


Transcribe: 100%|██████████| 379.73/379.73 [00:10<00:00, 35.92sec/s]
Adjustment: 100%|██████████| 379.63/379.63 [00:00<00:00, 34186.05sec/s]


Detected Language: russian


...

Transcribe: 100%|██████████| 404.79/404.79 [00:06<00:00, 63.03sec/s]
Adjustment: 100%|██████████| 402.4/402.4 [00:00<00:00, 110457.33sec/s]


Detected Language: russian


Transcribe: 100%|██████████| 387.08/387.08 [00:03<00:00, 96.83sec/s] 
Adjustment: 100%|██████████| 386.82/386.82 [00:00<00:00, 39453.36sec/s]


Detected Language: russian


Transcribe: 100%|██████████| 418.33/418.33 [00:05<00:00, 73.71sec/s]
Adjustment: 100%|██████████| 418.3/418.3 [00:00<00:00, 82540.34sec/s]


## Substantiating

#### Making JSONs from txts

In [154]:
my_dict = {}

json_txt_path = '/content/json_string_burnout.txt'
json_name_txt = json_txt_path.rsplit('/', 1)[-1]

with open(json_name_txt, 'r') as f:
    for line in f.readlines():
        feature, weight = line.replace(',', '.').split()
        my_dict[feature] = float(weight)

json_name = json_name_txt[:-3] + "json"

# 1. Write the dictionary to a file
with open(json_name, "w") as f:
    json.dump(my_dict, f, indent=4)

### Feature blocks

In [26]:
acoustic_block_list = '''
ACOUSTIC_F0final_sma_max ACOUSTIC_F0final_sma_mean ACOUSTIC_F0final_sma_min ACOUSTIC_F0final_sma_std ACOUSTIC_F0semitoneFrom27_5Hz_sma3nz_max ACOUSTIC_F0semitoneFrom27_5Hz_sma3nz_mean	ACOUSTIC_F0semitoneFrom27_5Hz_sma3nz_min
ACOUSTIC_F0semitoneFrom27_5Hz_sma3nz_std	ACOUSTIC_F1bandwidth_sma3nz_max	ACOUSTIC_F1bandwidth_sma3nz_mean ACOUSTIC_F1bandwidth_sma3nz_min	ACOUSTIC_F1bandwidth_sma3nz_std
ACOUSTIC_F1frequency_sma3nz_max	ACOUSTIC_F1frequency_sma3nz_mean	ACOUSTIC_F1frequency_sma3nz_min	ACOUSTIC_F1frequency_sma3nz_std	ACOUSTIC_F2bandwidth_sma3nz_max	ACOUSTIC_F2bandwidth_sma3nz_mean	ACOUSTIC_F2bandwidth_sma3nz_min
ACOUSTIC_F2bandwidth_sma3nz_std	ACOUSTIC_F2frequency_sma3nz_max	ACOUSTIC_F2frequency_sma3nz_mean	ACOUSTIC_F2frequency_sma3nz_min	ACOUSTIC_F2frequency_sma3nz_std	ACOUSTIC_F3bandwidth_sma3nz_max	ACOUSTIC_F3bandwidth_sma3nz_mean
ACOUSTIC_F3bandwidth_sma3nz_min	ACOUSTIC_F3bandwidth_sma3nz_std	ACOUSTIC_F3frequency_sma3nz_max	ACOUSTIC_F3frequency_sma3nz_mean	ACOUSTIC_F3frequency_sma3nz_min	ACOUSTIC_F3frequency_sma3nz_std	ACOUSTIC_HNRdBACF_sma3nz_max
ACOUSTIC_HNRdBACF_sma3nz_mean	ACOUSTIC_HNRdBACF_sma3nz_min	ACOUSTIC_HNRdBACF_sma3nz_std	ACOUSTIC_audspec_lengthL1norm_sma_max	ACOUSTIC_audspec_lengthL1norm_sma_mean	ACOUSTIC_audspec_lengthL1norm_sma_min	ACOUSTIC_audspec_lengthL1norm_sma_std
ACOUSTIC_jitterDDP_sma_max	ACOUSTIC_jitterDDP_sma_mean		ACOUSTIC_jitterDDP_sma_std	ACOUSTIC_jitterLocal_sma3nz_max	ACOUSTIC_jitterLocal_sma3nz_mean		ACOUSTIC_jitterLocal_sma3nz_std
ACOUSTIC_jitterLocal_sma_max	ACOUSTIC_jitterLocal_sma_mean		ACOUSTIC_jitterLocal_sma_std	ACOUSTIC_logHNR_sma_max	ACOUSTIC_logHNR_sma_mean	ACOUSTIC_logHNR_sma_min	ACOUSTIC_logHNR_sma_std	ACOUSTIC_logRelF0_H1_A3_sma3nz_max
ACOUSTIC_logRelF0_H1_A3_sma3nz_mean	ACOUSTIC_logRelF0_H1_A3_sma3nz_min	ACOUSTIC_logRelF0_H1_A3_sma3nz_std	ACOUSTIC_logRelF0_H1_H2_sma3nz_max	ACOUSTIC_logRelF0_H1_H2_sma3nz_mean	ACOUSTIC_logRelF0_H1_H2_sma3nz_min	ACOUSTIC_logRelF0_H1_H2_sma3nz_std
ACOUSTIC_mfcc1_sma3_max	ACOUSTIC_mfcc1_sma3_mean	ACOUSTIC_mfcc1_sma3_min	ACOUSTIC_mfcc1_sma3_std	ACOUSTIC_mfcc2_sma3_max	ACOUSTIC_mfcc2_sma3_mean	ACOUSTIC_mfcc2_sma3_min	ACOUSTIC_mfcc2_sma3_std	ACOUSTIC_mfcc3_sma3_max	ACOUSTIC_mfcc3_sma3_mean
ACOUSTIC_mfcc3_sma3_min	ACOUSTIC_mfcc3_sma3_std	ACOUSTIC_mfcc4_sma3_max	ACOUSTIC_mfcc4_sma3_mean	ACOUSTIC_mfcc4_sma3_min	ACOUSTIC_mfcc4_sma3_std	ACOUSTIC_mfcc_sma_10_max	ACOUSTIC_mfcc_sma_10_mean	ACOUSTIC_mfcc_sma_10_min	ACOUSTIC_mfcc_sma_10_std
ACOUSTIC_mfcc_sma_11_max	ACOUSTIC_mfcc_sma_11_mean	ACOUSTIC_mfcc_sma_11_min	ACOUSTIC_mfcc_sma_11_std	ACOUSTIC_mfcc_sma_12_max	ACOUSTIC_mfcc_sma_12_mean	ACOUSTIC_mfcc_sma_12_min	ACOUSTIC_mfcc_sma_12_std	ACOUSTIC_mfcc_sma_13_max
ACOUSTIC_mfcc_sma_13_mean	ACOUSTIC_mfcc_sma_13_min	ACOUSTIC_mfcc_sma_13_std	ACOUSTIC_mfcc_sma_14_max	ACOUSTIC_mfcc_sma_14_mean	ACOUSTIC_mfcc_sma_14_min	ACOUSTIC_mfcc_sma_14_std	ACOUSTIC_mfcc_sma_1_max	ACOUSTIC_mfcc_sma_1_mean
ACOUSTIC_mfcc_sma_1_min	ACOUSTIC_mfcc_sma_1_std	ACOUSTIC_mfcc_sma_2_max	ACOUSTIC_mfcc_sma_2_mean	ACOUSTIC_mfcc_sma_2_min	ACOUSTIC_mfcc_sma_2_std	ACOUSTIC_mfcc_sma_3_max	ACOUSTIC_mfcc_sma_3_mean	ACOUSTIC_mfcc_sma_3_min	ACOUSTIC_mfcc_sma_3_std
ACOUSTIC_mfcc_sma_4_max	ACOUSTIC_mfcc_sma_4_mean	ACOUSTIC_mfcc_sma_4_min	ACOUSTIC_mfcc_sma_4_std	ACOUSTIC_mfcc_sma_5_max	ACOUSTIC_mfcc_sma_5_mean	ACOUSTIC_mfcc_sma_5_min	ACOUSTIC_mfcc_sma_5_std	ACOUSTIC_mfcc_sma_6_max	ACOUSTIC_mfcc_sma_6_mean
ACOUSTIC_mfcc_sma_6_min	ACOUSTIC_mfcc_sma_6_std	ACOUSTIC_mfcc_sma_7_max	ACOUSTIC_mfcc_sma_7_mean	ACOUSTIC_mfcc_sma_7_min	ACOUSTIC_mfcc_sma_7_std	ACOUSTIC_mfcc_sma_8_max	ACOUSTIC_mfcc_sma_8_mean	ACOUSTIC_mfcc_sma_8_min	ACOUSTIC_mfcc_sma_8_std
ACOUSTIC_mfcc_sma_9_max	ACOUSTIC_mfcc_sma_9_mean	ACOUSTIC_mfcc_sma_9_min	ACOUSTIC_mfcc_sma_9_std	ACOUSTIC_pcm_RMSenergy_sma_max	ACOUSTIC_pcm_RMSenergy_sma_mean		ACOUSTIC_pcm_RMSenergy_sma_std
ACOUSTIC_pcm_fftMag_psySharpness_sma_max	ACOUSTIC_pcm_fftMag_psySharpness_sma_mean	ACOUSTIC_pcm_fftMag_psySharpness_sma_min	ACOUSTIC_pcm_fftMag_psySharpness_sma_std	ACOUSTIC_pcm_fftMag_spectralHarmonicity_sma_max
ACOUSTIC_pcm_fftMag_spectralHarmonicity_sma_mean		ACOUSTIC_pcm_fftMag_spectralHarmonicity_sma_std	ACOUSTIC_shimmerLocal_sma_max	ACOUSTIC_shimmerLocal_sma_mean	ACOUSTIC_shimmerLocal_sma_min
ACOUSTIC_shimmerLocal_sma_std	ACOUSTIC_shimmerLocaldB_sma3nz_max	ACOUSTIC_shimmerLocaldB_sma3nz_mean		ACOUSTIC_shimmerLocaldB_sma3nz_std'''.split()

asr_block_list = ['word_count', 'avg_interword_pause_ms', 'std_interword_pause_ms', 'avg_speech_rate_wps', 'std_speech_rate_wps', 'acc_speech_rate_wps', 'duration_sec', 'rms_energy']

emo_block_list = '''	EMO_Diminutive	EMO_Ethical	EMO_ExpressLex	EMO_General	EMO_Hyperbole	EMO_LoweredLex	EMO_Pragmatic	EMO_Rational	EMO_Sentimental	'''.split()

lex_block_list = '''LEX_Antithesis		LEX_Antonyms LEX_Attitude		LEX_Comparisons	LEX_EmotLex	LEX_IntelLex	LEX_MultiComp
LEX_Overlex				LEX_SynChain	LEX_SynRows34	LEX_TempLex	LEX_Tropes	LEX_Underlex 	LEX_WorkSyn'''.split()

mor_block_list = '''MOR_ActiveFreq	MOR_AdvMPC MOR_CoordConj	MOR_CoordDomin	MOR_GenderErr	MOR_ImperFreq	MOR_IntensPart	MOR_LimitPart	MOR_ModalFreq	MOR_PassiveFreq	MOR_RelPronConj	MOR_StateWords	MOR_SubordConj	MOR_SubordDomin'''.split()

ortpun_block_list = '''	ORT_AltRootVow		 	ORT_NotErr		ORT_TakZheErr
PUN_CommaDetach		PUN_Dash	PUN_EmoMarks	PUN_HomogErr		PUN_Quotes'''.split()

strsyn_block_list = '''STR_Argum	STR_Chaotic		STR_Coher	STR_Concise	STR_ConjLinks	STR_Deductive	STR_Digress
STR_EmClaim	STR_Inductive			STR_LogLink		STR_NoLogic	STR_SSC_Coherent SYN_AdjectPh	SYN_AsyndetFreq	SYN_ChtobyPref	SYN_ComplexSent	SYN_ConjFreq	SYN_ElipSent
SYN_HighSpread SYN_MixConst	SYN_NominalPh	SYN_Sogl	SYN_SoglDomin	SYN_Upravl	SYN_UpravlDomin	SYN_VerbalPh'''.split()

In [27]:
expert_block = ('Психолингвистика', list(json_strings_experts.keys()))

asr_block = ('Речевые характеристики', asr_block_list)

bert_block = ('Эмбеддинги BERT', [f'bert_{i + 1}' for i in range(1024)]) # хардкодом для скорости

acoustic_block = ('Акустика', acoustic_block_list)

emo_block = ('Эмоции и экспрессия', emo_block_list)
lex_block = ('Лексика и семантика', lex_block_list)

mor_block = ('Морфология', mor_block_list)
ortpun_block = ('Орфография и пунктуация', ortpun_block_list)

strsyn_block = ('Синтаксис', strsyn_block_list)

### Calculating importances

In [28]:
# подсчет вклада признаков
def calculate_block_importances(row: pd.Series,
                                block: list,
                                top_n: int = 3
                                ) -> pd.Series:

    # весовые коэффициенты для блока признаков
    block_weights = pd.Series({k: v for k, v in json_string_burnout.items() if k in block})
    # значения в блоке
    block_values = row[block]

    # итоговый вклад в оценку
    block_importances = (block_weights * block_values).\
                        sort_values(ascending=False, key=abs).\
                        head(top_n)

    # вернуть серию pandas
    return block_importances


### Interpretation loop

In [32]:
final_predictions_df

,file_name,burnout,burnout_final
0,20230904103009_Оператор12_muted.wav,-1.610026,0
1,20230927125324_Оператор12_muted.wav,-1.610026,0
2,20230927140858_Оператор12_muted.wav,-1.610026,0
3,20230927150013_Оператор12_muted.wav,-1.610026,0
4,20230928172132_Оператор12_muted.wav,-1.610026,0
5,20230928173608_Оператор12_muted.wav,-1.610026,0
6,20231006113013_Оператор12_muted.wav,-1.610026,0
7,20231019160503_Оператор12_muted.wav,-1.610026,0
8,20231019164521_Оператор12_muted.wav,-1.610026,0
9,20231023092551_Оператор12_muted.wav,-1.610026,0


In [31]:
final_interpretation_df = pd.read_excel('/content/result_full.xlsx')
final_interpretation_df

,file_name,agent_id,year,month,day,weekday,hour,text,word_count,avg_interword_pause_ms,...,farewell_flg,etiquette_present_flg,etiquette_absent_flg,participation_dominant_flg,distancing_dominant_flg,clarifying_absent_flg,m,f,burnout,burnout_final
0,20230904103009_Оператор12_muted.wav,_12,2023,9,4,Mon,10,Служба поддержки пользователей. Меня зовут Ана...,135,196,...,2.141660,1.949329,-0.949329,0.924801,0.057381,0.305739,1,0,-1.610026,0
1,20230927125324_Оператор12_muted.wav,_12,2023,9,27,Wed,12,"Служба поддержки пользователей, меня зовут Ана...",175,170,...,2.014764,1.753135,-0.753135,0.983027,0.006853,0.294120,1,0,-1.610026,0
2,20230927140858_Оператор12_muted.wav,_12,2023,9,27,Wed,14,Служба поддержки пользователей. Меня зовут Ана...,215,156,...,1.926226,1.760999,-0.760999,1.065340,-0.054354,0.150995,1,0,-1.610026,0
3,20230927150013_Оператор12_muted.wav,_12,2023,9,27,Wed,15,"Меня зовут Анастасия, добрый день. | К вам мог...",127,206,...,1.907264,1.649685,-0.649685,1.002892,-0.009662,0.270213,1,0,-1.610026,0
4,20230928172132_Оператор12_muted.wav,_12,2023,9,28,Thu,17,Служба поддержки пользователей. Меня зовут Ана...,268,280,...,2.064168,1.827915,-0.827915,1.081591,-0.063711,-0.005980,1,0,-1.610026,0
5,20230928173608_Оператор12_muted.wav,_12,2023,9,28,Thu,17,"Служба поддержки пользователей, меня зовут Ана...",277,234,...,1.915535,1.737694,-0.737694,0.904682,0.076116,0.161677,1,0,-1.610026,0
6,20231006113013_Оператор12_muted.wav,_12,2023,10,6,Fri,11,Служба поддержки пользователей. Меня зовут Ана...,72,173,...,2.099180,2.056094,-1.056094,0.931588,0.052649,0.411199,1,0,-1.610026,0
7,20231019160503_Оператор12_muted.wav,_12,2023,10,19,Thu,16,Служба поддержки пользователей. Меня зовут Ана...,390,220,...,2.085982,1.836418,-0.836418,0.992358,0.005826,0.104174,1,0,-1.610026,0
8,20231019164521_Оператор12_muted.wav,_12,2023,10,19,Thu,16,Служба поддержки пользователей. Меня зовут Ана...,139,192,...,1.902419,1.668687,-0.668687,1.011532,-0.014748,0.224346,1,0,-1.610026,0
9,20231023092551_Оператор12_muted.wav,_12,2023,10,23,Mon,9,"Служба поддержки пользователей, меня зовут Ана...",191,267,...,2.195646,1.978086,-0.978086,1.085726,-0.072233,0.235727,1,0,-1.610026,0


In [30]:
for i, row in final_interpretation_df.iterrows():
    print('*'*50)
    agent_sex = 'M' if row['m'] == 1 else 'F'

    print(f'{i} Файл:', row['file_name'])

    burnout_confidence, burnout_binarized = row['burnout'], row['burnout_final']

    if burnout_confidence < 0.5:
        burnout_confidence = 1 - burnout_confidence

    burnout_mapper = {1: 'Да', 0: 'Нет'}
    print(f'Выгорание: {burnout_mapper.get(burnout_binarized, "Не определено")} (уверенность: {burnout_confidence})')

    print(
        'Пол:', agent_sex
        )

    print('\nВклад признаков:')

    for j, block_tuple in enumerate([
        asr_block,
        # bert_block, # не интерпретируемый
        acoustic_block,
        emo_block,
        lex_block,
        mor_block,
        ortpun_block,
        strsyn_block,
        expert_block
        ], 1):
      block_name, block = block_tuple
      print('*'*50)
      print(f'{j}. {block_name}')
      print()
      print(calculate_block_importances(row, block, 1))
      print()

**************************************************
0 Файл: 20230904103009_Оператор12_muted.wav
Выгорание: Нет (уверенность: 2.610026187933757)
Пол: M

Вклад признаков:
**************************************************
1. Речевые характеристики

std_speech_rate_wps   -0.022991
dtype: object

**************************************************
2. Акустика



KeyError: "None of [Index(['ACOUSTIC_F0final_sma_max', 'ACOUSTIC_F0final_sma_mean',\n       'ACOUSTIC_F0final_sma_min', 'ACOUSTIC_F0final_sma_std',\n       'ACOUSTIC_F0semitoneFrom27_5Hz_sma3nz_max',\n       'ACOUSTIC_F0semitoneFrom27_5Hz_sma3nz_mean',\n       'ACOUSTIC_F0semitoneFrom27_5Hz_sma3nz_min',\n       'ACOUSTIC_F0semitoneFrom27_5Hz_sma3nz_std',\n       'ACOUSTIC_F1bandwidth_sma3nz_max', 'ACOUSTIC_F1bandwidth_sma3nz_mean',\n       ...\n       'ACOUSTIC_pcm_fftMag_spectralHarmonicity_sma_max',\n       'ACOUSTIC_pcm_fftMag_spectralHarmonicity_sma_mean',\n       'ACOUSTIC_pcm_fftMag_spectralHarmonicity_sma_std',\n       'ACOUSTIC_shimmerLocal_sma_max', 'ACOUSTIC_shimmerLocal_sma_mean',\n       'ACOUSTIC_shimmerLocal_sma_min', 'ACOUSTIC_shimmerLocal_sma_std',\n       'ACOUSTIC_shimmerLocaldB_sma3nz_max',\n       'ACOUSTIC_shimmerLocaldB_sma3nz_mean',\n       'ACOUSTIC_shimmerLocaldB_sma3nz_std'],\n      dtype='object', length=150)] are in the [index]"